# Recover mixture embeddings on `dcn_mix_exp.csv` using `gsolv_mix_comp` weights

This notebook uses:

- checkpoint: `chemprop/examples/MixtureDesign/weights/gsolv_mix_comp_mixture_context_solute_10k_fold_00_model.pt`
- prediction set: `datasets/fuel_ignition_numbers/processed_data/dcn_mix_exp.csv`

The source checkpoint was trained with a **solute + mixture** predictor input (202-D).
For this workflow we switch to a **mixture-only** architecture (101-D) end-to-end.

To keep the saved predictor usable without zero padding, the first predictor layer is remapped from 202-D to 101-D by taking the mixture half of the trained weight matrix.

Final predictor input is strictly:

`predictor_input = mixture_embedding`.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "scripts_training").exists() and (p / "chemprop").exists():
            return p
    raise RuntimeError(f"Could not locate repo root from {start}")


repo_root = find_repo_root(Path.cwd())
scripts_training_dir = repo_root / "scripts_training"
if str(scripts_training_dir) not in sys.path:
    sys.path.insert(0, str(scripts_training_dir))

import train_gnn_ffn as tgm
from utils.mixtures import collate_mixture

print(f"repo_root: {repo_root}")

repo_root: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix


In [2]:
# Configuration
mix_csv = repo_root / "datasets/fuel_ignition_numbers/processed_data/dcn_mix_exp.csv"
model_paths = [
    repo_root / "chemprop/examples/MixtureDesign/weights/gsolv_mix_comp_mixture_context_solute_10k_fold_00_model.pt",
]
output_dir = repo_root / "chemprop/examples/MixtureDesign/outputs/gsolv_mix_comp_to_dcn_mix"

batch_size = 128
aggregation = "weightedsum"
ffn_block_index = -1

# Encoder architecture switch: solute+mixture -> mixture-only
solute_component_index_encoder = -1

output_dir.mkdir(parents=True, exist_ok=True)
for p in model_paths:
    if not p.exists():
        raise FileNotFoundError(p)

print(f"input CSV:  {mix_csv}")
print(f"output dir: {output_dir}")
print(f"n_models:    {len(model_paths)}")

input CSV:  /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/datasets/fuel_ignition_numbers/processed_data/dcn_mix_exp.csv
output dir: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/outputs/gsolv_mix_comp_to_dcn_mix
n_models:    1


In [3]:
# Build inference dataset once (mixture-only architecture)
df = pd.read_csv(mix_csv)
all_data = tgm.build_all_data(
    df_mix=df.reset_index(drop=True),
    target_col="value",
    solute_component_index=solute_component_index_encoder,
)
n_components = len(all_data) - 1
dset = tgm.build_mixture_dataset(all_data, n_components=n_components, use_mixmp=True)
loader = DataLoader(dset, batch_size=batch_size, shuffle=False, collate_fn=collate_mixture)

# Dummy scaler only needed for model construction; predictor params are checkpoint-loaded.
y = pd.to_numeric(df["value"], errors="coerce").fillna(df["value"].mean()).to_numpy(dtype=float).reshape(-1, 1)
dummy_scaler = StandardScaler().fit(y)

print(f"rows: {len(df)} | n_components: {n_components}")

rows: 484 | n_components: 4


In [4]:
all_fold_predictions = []
summary_rows = []

for model_idx, model_path in enumerate(model_paths):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    use_mixmp = ckpt.get("mixmp") is not None

    # 1) Mixture-only encoder for embeddings
    mixture_model = tgm.build_model(
        n_components=n_components,
        scaler=dummy_scaler,
        aggregation=aggregation,
        solute_component_index=-1,
        use_mixmp=use_mixmp,
        x_d_dim=0,
    )
    mp_load = mixture_model.message_passing.load_state_dict(ckpt["message_passing"], strict=False)
    agg_load = mixture_model.agg.load_state_dict(ckpt["mixagg"], strict=False)
    mixture_model.eval()

    # 2) Mixture-only predictor head (101-D input)
    predictor_model = tgm.build_model(
        n_components=n_components,
        scaler=dummy_scaler,
        aggregation=aggregation,
        solute_component_index=-1,
        use_mixmp=use_mixmp,
        x_d_dim=0,
    )
    predictor_state = dict(ckpt["predictor"])
    first_key = "ffn.0.0.weight"
    w0 = predictor_state[first_key]
    d_mix = mixture_model.agg.output_dim
    d_target = predictor_model.predictor.ffn[0][0].weight.shape[1]
    if w0.shape[1] == 2 * d_mix and d_target == d_mix:
        # Original predictor input ordering is [solute, mixture]; keep mixture columns only.
        predictor_state[first_key] = w0[:, -d_mix:].clone()
    predictor_model.predictor.load_state_dict(predictor_state, strict=True)
    predictor_model.predictor.eval()

    preds_batches = []
    fp_batches = []
    enc_batches = []
    pred_in_batches = []

    with torch.no_grad():
        for batch in loader:
            bmgs, v_ds, x_d_batch, *_ = batch

            # Mixture-only learned embedding (101-D)
            z_mix = mixture_model.fingerprint(bmgs, v_ds, x_d_batch)

            z_pred = z_mix

            y_hat = predictor_model.predictor(z_pred)
            h_hat = predictor_model.predictor.encode(z_pred, ffn_block_index)

            preds_batches.append(y_hat.cpu().numpy().reshape(-1, 1))
            fp_batches.append(z_mix.cpu().numpy())
            pred_in_batches.append(z_pred.cpu().numpy())
            enc_batches.append(h_hat.cpu().numpy())

    fold_preds = np.concatenate(preds_batches, axis=0).reshape(-1)
    fold_fps = np.concatenate(fp_batches, axis=0)
    fold_pred_in = np.concatenate(pred_in_batches, axis=0)
    fold_enc = np.concatenate(enc_batches, axis=0)
    all_fold_predictions.append(fold_preds)

    fold_name = model_path.stem

    np.savez(
        output_dir / f"{fold_name}_embeddings.npz",
        row_index=np.arange(len(df), dtype=int),
        prediction=fold_preds,
        mixture_embedding=fold_fps,
        predictor_input=fold_pred_in,
        encoding=fold_enc,
    )

    pd.DataFrame(fold_fps, columns=[f"mix_emb_{i}" for i in range(fold_fps.shape[1])]).to_csv(
        output_dir / f"{fold_name}_mixture_embeddings.csv", index=False
    )
    pd.DataFrame(fold_pred_in, columns=[f"pred_in_{i}" for i in range(fold_pred_in.shape[1])]).to_csv(
        output_dir / f"{fold_name}_predictor_input.csv", index=False
    )
    pd.DataFrame(fold_enc, columns=[f"enc_{i}" for i in range(fold_enc.shape[1])]).to_csv(
        output_dir / f"{fold_name}_encodings.csv", index=False
    )

    summary_rows.append(
        {
            "model": str(model_path.relative_to(repo_root)),
            "name": fold_name,
            "n_rows": len(df),
            "mixture_embedding_dim": int(fold_fps.shape[1]),
            "predictor_input_dim": int(fold_pred_in.shape[1]),
            "encoding_dim": int(fold_enc.shape[1]),
            "mean_prediction": float(np.mean(fold_preds)),
            "std_prediction": float(np.std(fold_preds)),
            "mp_missing_keys": ";".join(mp_load.missing_keys),
            "mp_unexpected_keys": ";".join(mp_load.unexpected_keys),
            "agg_missing_keys": ";".join(agg_load.missing_keys),
            "agg_unexpected_keys": ";".join(agg_load.unexpected_keys),
        }
    )
    print(f"Saved {fold_name}: predictions + mixture embeddings + adapted encodings")

preds_mat = np.vstack(all_fold_predictions).T
preds_df = df.copy()
for i in range(preds_mat.shape[1]):
    preds_df[f"prediction_model_{i:02d}"] = preds_mat[:, i]
preds_df["prediction_mean"] = preds_mat.mean(axis=1)
preds_df["prediction_std"] = preds_mat.std(axis=1)
preds_df.to_csv(output_dir / "dcn_mix_exp_predictions_with_models.csv", index=False)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(output_dir / "embedding_export_summary.csv", index=False)

print("\nDone.")
print(f"Predictions table: {output_dir / 'dcn_mix_exp_predictions_with_models.csv'}")
print(f"Summary table:     {output_dir / 'embedding_export_summary.csv'}")

Saved gsolv_mix_comp_mixture_context_solute_10k_fold_00_model: predictions + mixture embeddings + adapted encodings

Done.
Predictions table: /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/outputs/gsolv_mix_comp_to_dcn_mix/dcn_mix_exp_predictions_with_models.csv
Summary table:     /Users/u0161682/Library/CloudStorage/OneDrive-KULeuven/Documents/ScientificOutput/CodeRepositories/GitLab/chempropmix/chemprop/examples/MixtureDesign/outputs/gsolv_mix_comp_to_dcn_mix/embedding_export_summary.csv
